In [1]:
# Install
!pip install pandas scikit-learn nltk matplotlib

# Imports
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

nltk.download('stopwords')

# Load dataset
df = pd.read_csv("QQ_Ar_Tafseer_training_8144.csv")

# Select text column
text_column = "text"   # <-- change if needed

# Arabic NLP cleaning
stop = set(stopwords.words("arabic"))

def clean_text(text):
    text = str(text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # remove diacritics
    text = re.sub(r'[إأآٱ]', 'ا', text)                # normalize Alef
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop and len(w) > 2]
    return " ".join(words)

df["clean_text"] = df[text_column].fillna("").apply(clean_text)

# TF-IDF
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = tfidf.fit_transform(df["clean_text"])

# K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# Show clusters
print(df["cluster"].value_counts().sort_index())

# Top words in each cluster
terms = tfidf.get_feature_names_out()

for i in range(k):
    words = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(terms[j] for j in words))

# PCA visualization
pca = PCA(n_components=2)
X2 = pca.fit_transform(X.toarray())

plt.figure(figsize=(8,5))
plt.scatter(X2[:,0], X2[:,1], c=df["cluster"], cmap="tab10")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Arabic Tafseer K-Means Clusters")
plt.show()

# Save result
df.to_csv("QQ_Ar_Tafseer_KMeans.csv", index=False, encoding="utf-8-sig")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyError: 'text'